# BIOTEQ Muscle Engine — Colab demo
Run **Runtime → Run all** on a CPU runtime. This notebook runs the Phase 1 `MuscleEngine` (`rules_v0.1`) on simulated bicep-curl data, displays derived metrics, plots sensor signals, and exports JSON.

Sections 1–5 demonstrate the deterministic rule-based engine. The final section trains and evaluates a separate public-data activity classifier. The demo threshold is arbitrary and does not diagnose injury. All example measurements and labels are simulated.

**Before using Colab:** commit and push the implementation and this notebook to GitHub, then open this `.ipynb` via Colab's GitHub tab. Changes that exist only on your computer are not available to Colab. For a private repository, upload a ZIP of your checkout using the option below; do not put access tokens in this notebook.

## 1. Locate the code and install dependencies
Choose `github` to download the public [b_smart repository](https://github.com/EEsc05/b_smart) as a ZIP with wget, or `upload_zip` to upload your local checkout (ZIP must include `bioteq/`, `requirements.txt`, and `examples/`; exclude `.venv` and secrets). Local Jupyter execution automatically reuses the checkout containing this notebook. No GPU is needed.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import zipfile
import tempfile

REPO_URL = "https://github.com/EEsc05/b_smart"
REPO_REF = "main"  # Published branch to download.
SOURCE = "github"  # Change to "upload_zip" for unpublished/local changes.

candidates = [Path.cwd(), Path.cwd().parent, Path("/content/b_smart")]
repo = next((p for p in candidates if (p / "bioteq/engine/muscle_engine.py").is_file()), None)
if repo is None:
    if SOURCE == "upload_zip":
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one repository ZIP.")
        destination = Path("/content/bioteq_upload").resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.infolist():
                if not (destination / item.filename).resolve().is_relative_to(destination):
                    raise ValueError("ZIP contains an unsafe path.")
            archive.extractall(destination)
        matches = list(destination.rglob("bioteq/engine/muscle_engine.py"))
        if len(matches) != 1:
            raise ValueError("ZIP must contain exactly one BIOTEQ implementation.")
        repo = matches[0].parents[2]
    elif SOURCE == "github":
        # A fresh directory lets reruns recover from incomplete earlier downloads.
        destination = Path(tempfile.mkdtemp(prefix="b_smart_"))
        archive_path = destination / "repository.zip"
        archive_url = f"{REPO_URL}/archive/refs/heads/{REPO_REF}.zip"
        result = subprocess.run(
            ["wget", "--quiet", "--timeout=30", "--tries=2", archive_url, "-O", str(archive_path)],
            capture_output=True, text=True,
        )
        if result.returncode:
            raise RuntimeError(
                f"Could not download {archive_url}. Check that the repository and "
                "branch are public, or set SOURCE = 'upload_zip'. "
                f"wget exit code: {result.returncode}. {result.stderr.strip()}"
            )
        with zipfile.ZipFile(archive_path) as archive:
            for item in archive.infolist():
                if not (destination / item.filename).resolve().is_relative_to(destination):
                    raise ValueError("ZIP contains an unsafe path.")
            archive.extractall(destination)
        matches = list(destination.rglob("bioteq/engine/muscle_engine.py"))
        if len(matches) != 1:
            raise RuntimeError(
                "The downloaded branch must contain one BIOTEQ implementation. "
                "Push bioteq/, requirements.txt, and examples/ to b_smart, "
                "or use SOURCE = 'upload_zip' with your local checkout."
            )
        repo = matches[0].parents[2]
    else:
        raise ValueError("SOURCE must be github or upload_zip.")
repo = repo.resolve()
if not (repo / "bioteq/engine/muscle_engine.py").is_file():
    raise RuntimeError("The selected GitHub version lacks the engine. Push your changes or use upload_zip.")
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
# Set this variable only for local validation in an already prepared environment.
if os.environ.get("BIOTEQ_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "matplotlib>=3.8,<4"])
print("Using repository:", repo)

## 2. Load a session and run the engine
The supplied fixture contains ten repetitions over 30 seconds, EMG sampled every 250 ms, motion every 100 ms, and heart rate/NIRS every second. Default rules produce no demo warning. A second pass explicitly enables the software demonstration threshold.

To use your own simulator CSV, change `CSV_PATH`. It must follow `SensorReading` and contain one session and exercise. Do not supply event labels as sensor inputs.

In [ ]:
import json
from IPython.display import display, HTML, FileLink
from bioteq.ingestion.reader import SensorReader
from bioteq.engine.muscle_engine import MuscleEngine

CSV_PATH = repo / "examples/sample_session.csv"
readings = SensorReader().read_csv(CSV_PATH)
output = MuscleEngine().process_session(readings)
demo_output = MuscleEngine(config_path=repo / "bioteq/config/demo_thresholds.yaml").process_session(readings)
print(f"Session: {output.session_id} | exercise: {output.exercise}")
print(f"Data source: {output.data_source} | engine: {output.model_version}")
print(f"Readings: {len(readings):,} | duration: {(output.end_ms-output.start_ms)/1000:g} s")

## 3. Derived metrics and rule outputs
Missing measurements display as unavailable. EMG normalization and NIRS baseline changes require explicit references, so they remain null in this default demo. Heart-rate recovery requires an explicit rest interval. The 38% planted difference uses the left side as denominator; symmetric imbalance uses the mean of both sides.

In [ ]:
from html import escape

def table(headers, rows):
    def cell(value):
        if value is None:
            return "unavailable"
        if isinstance(value, float):
            return f"{value:.3f}"
        return escape(str(value))
    display(HTML("<table><thead><tr>" + "".join(f"<th>{escape(h)}</th>" for h in headers) +
                 "</tr></thead><tbody>" + "".join("<tr>" + "".join(f"<td>{cell(v)}</td>" for v in row) +
                 "</tr>" for row in rows) + "</tbody></table>"))

m = output.metrics
table(["Metric", "Value", "Unit"], [
    ("Repetitions", m.movement.rep_count, "count"),
    ("Average peak-to-peak repetition duration", m.movement.average_rep_duration_s, "s"),
    ("Repetition duration variability", m.movement.movement_variability_cv, "CV"),
    ("Mean heart rate", m.cardiovascular.heart_rate_bpm, "bpm"),
    ("Heart-rate recovery", m.cardiovascular.heart_rate_recovery_bpm, "bpm"),
])
table(["Muscle", "Side", "RMS (mV)", "Mean absolute (mV)", "Normalized activation (%)"],
      [(x.muscle_site, x.side, x.rms_mv, x.mean_absolute_mv, x.normalized_activation_pct) for x in m.muscles])
table(["Muscle", "Symmetric imbalance (%)", "Right greater than left (%)"],
      [(x.muscle_site, x.imbalance_pct, x.right_greater_than_left_pct) for x in m.bilateral])
table(["Muscle", "Side", "Mean oxygenation (%)", "Baseline change (percentage points)"],
      [(x.muscle_site, x.side, x.smo2_pct, x.change_percentage_points) for x in m.oxygenation])
print("Quality flags:", output.quality_flags or "none")
print("Default rule warnings:", len(output.warnings))
print("Demo rule warnings:", len(demo_output.warnings))
for warning in demo_output.warnings:
    print(f"{warning.code}: {warning.message} Evaluated window: {warning.start_ms}–{warning.end_ms} ms.")

## 4. Plot the observed signals and derived EMG RMS
Motion magnitude uses complete x/y/z readings at identical timestamps. Sensor channels retain their original timestamps; no missing values are interpolated. The fixture's constant EMG, heart-rate, and NIRS signals are deliberate software inputs, not realistic physiological waveforms. Rep counting depends on gravity, orientation, and exercise.

In [ ]:
from collections import defaultdict
from math import sqrt
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
fig.suptitle(f"BIOTEQ — {output.session_id} ({output.data_source})")

def signal_plot(ax, channel, unit, ylabel):
    groups = defaultdict(list)
    for r in sorted(readings, key=lambda r: r.timestamp_ms):
        if r.channel == channel and r.unit == unit and r.value is not None:
            groups[(r.muscle_site, r.side)].append((r.timestamp_ms/1000, r.value))
    for (site, side), points in groups.items():
        times, values = zip(*points)
        ax.plot(times, values, label=" ".join(x for x in (site, side) if x) or channel.upper())
    if groups:
        ax.legend(fontsize=8)
    else:
        ax.text(.5, .5, "No compatible measurements", ha="center", transform=ax.transAxes)
    ax.set(xlabel="Time (s)", ylabel=ylabel)
    ax.grid(alpha=.25)

signal_plot(axes[0, 0], "emg", "mV", "EMG amplitude (mV)")
axes[0, 0].set_title("Observed EMG")
frames = defaultdict(dict)
for r in readings:
    if r.channel == "accelerometer" and r.value is not None:
        frames[r.timestamp_ms][r.axis] = r.value
motion = sorted((t/1000, sqrt(sum(v[a]**2 for a in ("x", "y", "z"))))
                for t, v in frames.items() if all(a in v for a in ("x", "y", "z")))
if motion:
    times, values = zip(*motion)
    axes[0, 1].plot(times, values, color="tab:orange")
else:
    axes[0, 1].text(.5, .5, "No complete XYZ frames", ha="center", transform=axes[0, 1].transAxes)
axes[0, 1].set(title=f"Acceleration magnitude — {m.movement.rep_count} detected reps",
               xlabel="Time (s)", ylabel="Magnitude (m/s²)")
axes[0, 1].grid(alpha=.25)
signal_plot(axes[1, 0], "ecg", "bpm", "Heart rate (bpm)")
axes[1, 0].set_title("Supplied heart rate")
signal_plot(axes[1, 1], "nirs", "percent", "Muscle oxygen saturation (%)")
axes[1, 1].set_title("Observed NIRS")
plt.show()

if m.muscles:
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.bar([f"{x.muscle_site}\n{x.side or 'unpaired'}" for x in m.muscles],
           [x.rms_mv for x in m.muscles], color=["tab:blue" if x.side == "left" else "tab:orange" for x in m.muscles])
    ax.set(title="Derived whole-window EMG RMS", ylabel="RMS amplitude (mV)")
    ax.grid(axis="y", alpha=.25)
    plt.show()

## Muscle activation maps
The diagram highlights only recorded biceps and trapezius regions; other regions stay gray. These are schematic sensor locations, not anatomical localization or predictions from the activity classifier. All signals in the default example are simulated.

The default color scale shows **raw EMG RMS in mV**, which cannot establish comparable effort across muscles. Set `NORMALIZE_EMG = True` and supply positive reference RMS measurements to show percent of each reference. Use a validated MVC protocol before interpreting values as percent MVC. Channels without references stay gray in normalized mode; values can exceed 100%.

The time heatmap uses non-overlapping one-second bins, without interpolation. Gray means unavailable; measured zero has the lightest color. The supplied signals have nearly constant amplitudes, so flat colors are expected. Both figures use the same scale.


In [ ]:
from bioteq.visualization.muscles import plot_muscles

NORMALIZE_EMG = False
EMG_REFERENCES = {}  # e.g. {("biceps", "right"): measured_reference_rms_mv}
EMG_BIN_MS = 1000

if any(r.channel == "emg" and r.unit == "mV" for r in readings):
    body_figure, time_figure, muscle_map_info = plot_muscles(
        readings, bin_ms=EMG_BIN_MS, references=EMG_REFERENCES, normalized=NORMALIZE_EMG)
    print("Source:", output.data_source, "| Color scale:", muscle_map_info["unit"])
    print(muscle_map_info["note"])
    if muscle_map_info["unmapped_sites"]:
        print("Sites included in time heatmap but not body schematic:", muscle_map_info["unmapped_sites"])
    plt.show()
    table(["Muscle", "Side", muscle_map_info["unit"]],
          [(site, side, value) for (site, side), value in muscle_map_info["session_rms"].items()])
    muscle_export_dir = repo / "notebook_outputs"
    muscle_export_dir.mkdir(exist_ok=True)
    for name, figure in [("muscle_body_map", body_figure), ("muscle_time_heatmap", time_figure)]:
        for extension in ("png", "svg"):
            target = muscle_export_dir / f"{name}.{extension}"
            figure.savefig(target, dpi=180, bbox_inches="tight")
            display(FileLink(str(target)))
else:
    print("No EMG signals in mV: muscle maps unavailable. Activity labels cannot supply muscle activation.")


## 5. Inspect and export structured output
The default and demo results remain separate. Warning bounds describe the evaluated window, not a localized onset. `FileLink` works in Jupyter; the optional download cell uses Colab's download interface.

In [ ]:
print(output.model_dump_json(indent=2))
export_dir = repo / "notebook_outputs"
export_dir.mkdir(exist_ok=True)
for name, result in [("session_output.json", output), ("demo_output.json", demo_output)]:
    path = export_dir / name
    path.write_text(result.model_dump_json(indent=2) + "\n")
    display(FileLink(str(path)))

In [ ]:
DOWNLOAD_JSON = False  # Set True to download both result files in Colab.
if DOWNLOAD_JSON:
    from google.colab import files
    for name in ("session_output.json", "demo_output.json"):
        files.download(str(export_dir / name))

## 6. Sample-fixture checks
These checks verify the deterministic software example only. They are not model accuracy scores or public-data validation. Checks skip automatically for a different CSV.

In [ ]:
if CSV_PATH.resolve() == (repo / "examples/sample_session.csv").resolve():
    import math
    assert output.metrics.movement.rep_count == 10
    assert math.isclose(output.metrics.movement.average_rep_duration_s, 3.0)
    trapezius = next(x for x in m.bilateral if x.muscle_site == "trapezius")
    assert math.isclose(trapezius.right_greater_than_left_pct, 38.0)
    assert not output.warnings
    assert any(w.code == "possible_compensation" for w in demo_output.warnings)
    print("Passed: 10 repetitions, 3-second cadence, 38% right-side difference, and demo-only rule flag.")
else:
    print("Custom session: sample-specific assertions skipped.")

## 7. Public-data ML evaluation
This separate random-forest model recognizes 12 activities from wearable accelerometers. It does not assess curl technique, compensation, fatigue, or injury.

Data attribution: Banos, Garcia & Saez (2014), [MHEALTH, UCI](https://doi.org/10.24432/C5TW22), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The first recorded run achieved **96.23% accuracy**, **96.34% balanced accuracy**, and **96.40% macro F1** on 3,313 windows, versus **8.99%** for a majority-class baseline.

By default, display the saved evaluation report immediately. Set `RETRAIN = True` to download about 72 MB and repeat all ten participant folds, then train a local model. Each participant is excluded from training when evaluated. Package versions can affect regenerated results.


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, FileLink

RETRAIN = False
report_path = repo / "reports/activity_metrics.json"
if RETRAIN:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ml.txt"])
    from bioteq.ml.activity import train
    report = train()
elif report_path.is_file():
    report = json.loads(report_path.read_text())
else:
    raise FileNotFoundError("The saved report is missing. Upload the updated checkout or set RETRAIN = True.")
print("Results:", "new evaluation" if RETRAIN else "saved evaluation (not retrained this session)")
print(f"{report['windows']:,} eligible windows | {len(report['folds'])} held-out participants")
print("Source:", report["provenance"]["source"])
print("Recorded package versions:", report["versions"])


### Overall performance
Accuracy counts correctly classified windows. Balanced accuracy averages class recall; macro F1 gives each class equal weight. Higher is better for these three metrics. Lower log loss and Brier score mean better probability predictions. Forest probabilities are **uncalibrated**.


In [ ]:
metrics = report["pooled_out_of_fold"]
percent = lambda value: f"{100 * value:.2f}%"
table(["Metric", "Held-out result"], [
    ("Accuracy", percent(metrics["accuracy"])),
    ("Balanced accuracy", percent(metrics["balanced_accuracy"])),
    ("Macro F1", percent(metrics["macro_f1"])),
    ("Majority baseline accuracy", percent(report["majority_baseline_accuracy"])),
    ("Log loss (lower is better)", metrics["log_loss"]),
    ("Multiclass Brier score (lower is better)", metrics["multiclass_brier"]),
])
fig, ax = plt.subplots(figsize=(7, 3))
values = [report["majority_baseline_accuracy"], metrics["accuracy"]]
bars = ax.barh(["Majority baseline", "Random forest"], np.array(values) * 100, color=["gray", "teal"])
ax.bar_label(bars, fmt="%.2f%%", padding=4)
ax.set(xlim=(0, 110), xlabel="Held-out window accuracy (%)", title="Baseline comparison")
plt.tight_layout()
plt.show()


### Results by participant and activity
The participant chart shows how performance varies across people. Precision measures how often a predicted activity is correct; recall measures how many examples of that activity were found. Support is the number of evaluated windows.


In [ ]:
folds = report["folds"]
table(["Held-out participant", "Windows", "Accuracy", "Macro F1"], [
    (f["test_subject"], f["test_windows"], percent(f["accuracy"]), percent(f["macro_f1"])) for f in folds])
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar([str(f["test_subject"]) for f in folds], [100*f["accuracy"] for f in folds], color="teal")
ax.axhline(100*metrics["accuracy"], color="black", linestyle="--", label="Pooled accuracy")
ax.set(ylim=(0, 105), xlabel="Held-out participant", ylabel="Accuracy (%)", title="Generalization across participants")
ax.legend()
plt.tight_layout()
plt.show()
table(["Activity", "Precision", "Recall", "F1", "Windows"], [
    (label.replace("_", " "), percent(report["per_class"][label]["precision"]),
     percent(report["per_class"][label]["recall"]), percent(report["per_class"][label]["f1-score"]),
     int(report["per_class"][label]["support"])) for label in report["class_order"]])


### Confusion matrix
Rows are actual activities; columns are predictions. Diagonal cells are correct. Percentages are normalized within each actual activity, making classes with different sample counts easier to compare.


In [ ]:
matrix = np.asarray(report["confusion_matrix"], dtype=float)
row_counts = matrix.sum(axis=1, keepdims=True)
normalized = np.divide(100*matrix, row_counts, out=np.zeros_like(matrix), where=row_counts != 0)
labels = [label.replace("_", " ") for label in report["class_order"]]
fig, ax = plt.subplots(figsize=(11, 9))
plot = ax.imshow(normalized, vmin=0, vmax=100, cmap="Blues")
for i in range(len(labels)):
    for j in range(len(labels)):
        if matrix[i, j]:
            ax.text(j, i, f"{normalized[i,j]:.0f}%", ha="center", va="center",
                    color="white" if normalized[i,j] > 50 else "black", fontsize=8)
ax.set(xticks=range(len(labels)), yticks=range(len(labels)), xticklabels=labels,
       yticklabels=labels, xlabel="Predicted activity", ylabel="Actual activity",
       title="Held-out predictions (% of actual activity)")
plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
fig.colorbar(plot, ax=ax, label="Percent of actual activity")
plt.tight_layout()
plt.show()
display(FileLink(str(report_path)))


### Interpretation and optional inference
These scores describe activity recognition in this dataset. Only ten participants were evaluated, and windows containing transitions or unknown activities were excluded. Continuous real-world use may perform worse. These are not repetition-counting or compensation-detection scores.

Inference needs a 100 × 9 acceleration window at 50 Hz with the original sensor placements. Set `RETRAIN = True` above to create the local model and example. The example comes from training data and is only a demonstration; its prediction does not contribute to the reported test scores.


In [ ]:
if RETRAIN:
    import joblib
    from bioteq.ml.activity import predict_window
    bundle = joblib.load(repo / "artifacts/activity/model.joblib")
    example = json.loads((repo / "artifacts/activity/example_window.json").read_text())
    prediction = predict_window(bundle, example)
    print("Training-example prediction:", prediction["activity"])
    table(["Activity", "Uncalibrated probability"], [
        (label, percent(probability)) for label, probability in
        sorted(prediction["probabilities"].items(), key=lambda item: item[1], reverse=True)])
else:
    print("Saved evaluation displayed. Set RETRAIN = True to train and run inference.")
